# 04c · Pipeline của một PINN gồm những gì?

**Điều kiện trước:** đã chạy [04a](04a-neural-network.ipynb) và [04b](04b-autograd.ipynb). Bài này đi qua một vòng PINN rất nhỏ: **bài toán → mạng → điểm tính → residual → loss → train → verify**. Ta giải dầm chuẩn hóa $u^{(4)}(x)=1$, $u(0)=u(1)=u''(0)=u''(1)=0$. Nghiệm đúng chỉ dùng ở bước verify.

## 1. Tạo mạng và tọa độ

`x_c`: điểm collocation trong miền để kiểm tra PDE. `x_b`: hai đầu để kiểm tra điều kiện biên. Cả hai phải có `requires_grad=True` vì sẽ lấy đạo hàm theo tọa độ.

In [ ]:
import torch
from torch import nn

torch.manual_seed(7)
torch.set_num_threads(1)
model = nn.Sequential(nn.Linear(1, 16), nn.Tanh(),
                      nn.Linear(16, 16), nn.Tanh(),
                      nn.Linear(16, 1)).double()

x_c = torch.linspace(0.05, 0.95, 16, dtype=torch.float64).reshape(-1, 1)
x_c.requires_grad_(True)
x_b = torch.tensor([[0.0], [1.0]], dtype=torch.float64, requires_grad=True)
print("điểm miền:", x_c.shape, "điểm biên:", x_b.shape)

## 2. Viết residual và loss trước khi train

Trên miền, $r=u_\theta^{(4)}-1$ và $L_{\rm PDE}=\operatorname{mean}(r^2)$. Ở biên, $L_{\rm BC}=\operatorname{sum}(u_b^2+u_b''{}^2)$. Mạng **chưa biết** nghiệm đúng ở hai bước này.

In [ ]:
def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

u = model(x_c)
u1 = grad(u, x_c)
u2 = grad(u1, x_c)
u3 = grad(u2, x_c)
u4 = grad(u3, x_c)
loss_pde = ((u4 - 1)**2).mean()

u_b = model(x_b)
u_b1 = grad(u_b, x_b)
u_b2 = grad(u_b1, x_b)
loss_bc = (u_b**2 + u_b2**2).sum()
print("trước train: PDE", loss_pde.item(), "BC", loss_bc.item())

## 3. Một bước train

`zero_grad()` xóa gradient trọng số cũ; `backward()` tính gradient theo trọng số; `step()` cập nhật trọng số. Sau bước này, các trọng số của `model` đã đổi. Ta tính lại loss thay vì dùng giá trị cũ ở ô trên.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
optimizer.zero_grad()
loss = loss_pde + 10*loss_bc
loss.backward()
optimizer.step()
print("loss trước bước cập nhật:", loss.item())

## 4. Lặp các bước train

Mỗi vòng phải **forward lại** với trọng số vừa cập nhật, tính lại residual và loss, rồi cập nhật tiếp. Không được tái sử dụng `loss` của ô trước. Mã bên dưới lặp 150 lần; không cần viết lớp Python riêng.

In [ ]:
loss_history = []
for step in range(150):
    optimizer.zero_grad()
    u = model(x_c)
    u1 = grad(u, x_c)
    u2 = grad(u1, x_c)
    u3 = grad(u2, x_c)
    u4 = grad(u3, x_c)
    loss_pde = ((u4 - 1)**2).mean()

    u_b = model(x_b)
    u_b1 = grad(u_b, x_b)
    u_b2 = grad(u_b1, x_b)
    loss_bc = (u_b**2 + u_b2**2).sum()

    loss = loss_pde + 10*loss_bc
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

print("loss đầu/cuối:", loss_history[0], loss_history[-1])

## 5. Verify độc lập

Tạo 101 điểm **mới**. Chỉ bây giờ dùng nghiệm đúng $u_*(x)=(x^4-2x^3+x)/24$. Sai số tương đối không giống loss train: nó so giá trị nghiệm trên lưới kiểm chứng.

In [ ]:
x_test = torch.linspace(0, 1, 101, dtype=torch.float64).reshape(-1, 1)
u_pred = model(x_test)
u_exact = (x_test**4 - 2*x_test**3 + x_test) / 24
error = torch.sqrt(((u_pred-u_exact)**2).sum() / (u_exact**2).sum())
print("sai số chuyển vị tương đối:", error.item())
print("u ở hai đầu:", model(x_b).detach().reshape(-1).tolist())

**Tự làm:** chỉ đúng dòng của bảy bước pipeline; đổi **một** trong hai yếu tố: 16 điểm miền thành 32, hoặc 150 bước thành 300. So loss và sai số kiểm chứng. **Sản phẩm:** một bảng hai cấu hình và lời giải thích vì sao loss train khác sai số nghiệm.

**Tiếp theo:** [04 · notebook dầm đầy đủ](04-torch-pinn.ipynb) bổ sung mômen, hình và thí nghiệm; [05](05-beam-fem-to-pinn.ipynb) so FEM–PINN.